# Practical Retrieval-Augmented Generation

## Workshop overview

This notebook builds a complete retrieval-augmented generation system from first principles.

The system has three major parts:

```text
PART 1 — INGESTION
documents -> chunks -> embeddings -> ChromaDB

PART 2 — RAG
question -> retrieval -> grounded prompt
         -> structured answer -> citation validation

PART 3 — TESTING & EVALUATION
retrieval quality -> answer quality -> citation quality
                  -> end-to-end behaviour
```

The notebook contains the full implementation. The clean Python modules in `src/`
mirror the same functions after they have been understood here.

## System architecture

```mermaid
flowchart LR
    subgraph Ingestion["Part 1 — Ingestion"]
        A[PDF documents] --> B[Load pages]
        B --> C[Split into chunks]
        C --> D[Generate embeddings]
        D --> E[(ChromaDB)]
    end

    subgraph RAG["Part 2 — RAG"]
        Q[User question] --> F[Generate query embedding]
        F --> G[Retrieve top-k chunks]
        E --> G
        G --> H[Construct grounded prompt]
        H --> I[OpenAI Agent]
        I --> J[Structured answer]
        J --> K[Validate citations]
    end

    subgraph Evaluation["Part 3 — Testing & Evaluation"]
        G --> L[Retrieval evaluation]
        J --> M[Generation evaluation]
        K --> N[Citation evaluation]
    end
```

## Environment and configuration

Configuration values are grouped in one place so the main design decisions remain visible.

The embedding model is local. OpenAI is used only for the final answer-generation stage.

In [1]:
from __future__ import annotations

import os
import shutil
from pathlib import Path
from typing import Any

import chromadb
import numpy as np
from agents import Agent, Runner
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pydantic import BaseModel, Field
from sentence_transformers import SentenceTransformer

from pprint import pprint


# Resolve the project root whether the notebook is opened from the project
# directory or from the notebooks directory.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

load_dotenv(PROJECT_ROOT / ".env")

DATA_DIR = PROJECT_ROOT / "data" / "knowledge_base"
CHROMA_DIR = PROJECT_ROOT / "chroma_db"

COLLECTION_NAME = "asterlane_policies"
EMBEDDING_MODEL_NAME = os.getenv(
    "EMBEDDING_MODEL",
    "sentence-transformers/all-MiniLM-L6-v2",
)
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

CHUNK_SIZE = 700
CHUNK_OVERLAP = 100
TOP_K = 4

print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)
print("Chroma directory:", CHROMA_DIR)
print("Embedding model:", EMBEDDING_MODEL_NAME)
print("OpenAI model:", OPENAI_MODEL)


/var/folders/c7/hvdqzbm91lb35c9cj67ptwk00000gn/T/ipykernel_98691/2709474377.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader


Project root: /Users/saed/Development/teaching/courses/practical_ai_engineering/day_4_practical_rag
Data directory: /Users/saed/Development/teaching/courses/practical_ai_engineering/day_4_practical_rag/data/knowledge_base
Chroma directory: /Users/saed/Development/teaching/courses/practical_ai_engineering/day_4_practical_rag/chroma_db
Embedding model: sentence-transformers/all-MiniLM-L6-v2
OpenAI model: gpt-4o-mini


# Part 1 — Ingestion Pipeline

The ingestion pipeline transforms unstructured PDF documents into a searchable vector collection.

The stages are:

1. load and inspect the source documents;
2. split pages into retrieval-sized chunks;
3. generate local text embeddings;
4. store text, vectors and metadata in ChromaDB.

## 1.1 Loading source documents

Each loaded page is represented as a LangChain `Document` containing:

- `page_content`: the extracted text;
- `metadata`: information such as source filename and page number.

The function below normalises metadata and removes empty pages.

In [2]:
# Inspect the core function of the loding function
raw_documents = PyPDFDirectoryLoader(str(DATA_DIR)).load()

# It does the following:
# 1. Finds PDF files inside DATA_DIR.
# 2. Extracts text from each PDF using pypdf.
# 3. Creates roughly one Document object per PDF page.
# 4. Combines all those page-level documents into one list.

In [3]:
type(raw_documents),len(raw_documents)

(list, 9)

In [4]:
# Information such as source filename and page number
raw_documents[0].metadata

{'producer': 'LibreOffice 25.2.3.2 (X86_64) / LibreOffice Community',
 'creator': 'Writer',
 'creationdate': '2026-06-14T11:46:37+00:00',
 'title': 'Support Escalation Guide',
 'author': 'AsterLane Delivery Services Ltd',
 'subject': 'Fictional internal knowledge base for RAG training',
 'keywords': 'escalation, safety, fraud, account, security, privacy, complaints',
 'source': '/Users/saed/Development/teaching/courses/practical_ai_engineering/day_4_practical_rag/data/knowledge_base/support_escalation_guide.pdf',
 'total_pages': 3,
 'page': 0,
 'page_label': '1'}

In [5]:
# Extracted text from one PDF page
raw_documents[0].page_content

'AsterLane Delivery Services Ltd | ALS-SUP-002 | Version 1.4\nInternal - Restricted | Page 1\nAsterLane Delivery Services Ltd\nSupport Escalation Guide\nInternal Knowledge Base - Fictional Training Material\nMetadata field Value\nDocument ID ALS-SUP-002\nVersion 1.4\nEffective date 1 June 2026\nReview date 1 September 2026\nDocument owner Support Quality and Risk\nApproved by Customer Operations Director\nAudience Customer Support, Courier Support, Team Leaders\nRegion United Kingdom\nClassification Internal - Restricted\nStatus Active\nTags escalation, safety, fraud, account security, privacy, \ncomplaints\nTraining notice: This is fictional material created for teaching retrieval-augmented generation.'

In [6]:
def load_documents(data_dir: Path) -> list[Document]:
    """Load PDF pages and normalise source and page metadata."""

    if not data_dir.exists():
        raise FileNotFoundError(f"Data directory does not exist: {data_dir}")

    # PyPDFDirectoryLoader loads every PDF in the directory.
    raw_documents = PyPDFDirectoryLoader(str(data_dir)).load()

    if not raw_documents:
        raise ValueError(f"No PDF pages were loaded from {data_dir}")

    documents: list[Document] = []

    for document in raw_documents:
        text = document.page_content.strip()

        # Empty pages contain no searchable evidence and are excluded.
        if not text:
            continue

        # Store only the filename rather than the complete machine-specific path.
        source = Path(str(document.metadata.get("source", "unknown"))).name

        # LangChain uses zero-based page indices. Convert to one-based page
        # numbers so citations match the numbering visible in the PDF.
        page = int(document.metadata.get("page", 0)) + 1

        documents.append(
            Document(
                page_content=text,
                metadata={
                    "source": source,
                    "page": page,
                },
            )
        )

    if not documents:
        raise ValueError("The PDFs contained no extractable text.")

    return documents

In [7]:
documents = load_documents(DATA_DIR)

print(f"Loaded {len(documents)} pages")
print("Sources:", sorted({doc.metadata["source"] for doc in documents}))

sample_page = documents[0]
print("\nMetadata:", sample_page.metadata)
print("\nText preview:\n", sample_page.page_content[:1000])

Loaded 9 pages
Sources: ['courier_operations_handbook.pdf', 'customer_delivery_and_refund_policy.pdf', 'support_escalation_guide.pdf']

Metadata: {'source': 'support_escalation_guide.pdf', 'page': 1}

Text preview:
 AsterLane Delivery Services Ltd | ALS-SUP-002 | Version 1.4
Internal - Restricted | Page 1
AsterLane Delivery Services Ltd
Support Escalation Guide
Internal Knowledge Base - Fictional Training Material
Metadata field Value
Document ID ALS-SUP-002
Version 1.4
Effective date 1 June 2026
Review date 1 September 2026
Document owner Support Quality and Risk
Approved by Customer Operations Director
Audience Customer Support, Courier Support, Team Leaders
Region United Kingdom
Classification Internal - Restricted
Status Active
Tags escalation, safety, fraud, account security, privacy, 
complaints
Training notice: This is fictional material created for teaching retrieval-augmented generation.


## 1.2 Document inspection

Ingestion quality should be checked before any embedding or retrieval work begins.

Typical checks include:

- no empty pages;
- valid source filenames;
- correct page numbering;
- readable extracted text.

In [8]:
def inspect_documents(documents: list[Document]) -> dict[str, Any]:
    """Return simple diagnostics for the loaded document collection."""
    
    number_of_pages = len(documents)


    # count empty pages
    empty_pages = []
    for document in documents:
        if not document.page_content.strip():
            empty_pages.append(document)
    
    number_of_empty_pages = len(empty_pages)


    # sources and characters
    sources = []
    character_counts = []
    for document in documents:
        source = str(document.metadata["source"])
        sources.append(source)

        number_of_characters = len(document.page_content)
        character_counts.append(number_of_characters)

    unique_sources = set(sources)
    sorted_sources = sorted(unique_sources)
    average_characters_per_page = round(np.mean(character_counts),1)


    results = {
        "pages": number_of_pages,
        "sources": sorted_sources,
        "empty_pages": number_of_empty_pages,
        "average_characters_per_page": average_characters_per_page,
    }

    return results


In [10]:
document_report = inspect_documents(documents)
document_report

{'pages': 9,
 'sources': ['courier_operations_handbook.pdf',
  'customer_delivery_and_refund_policy.pdf',
  'support_escalation_guide.pdf'],
 'empty_pages': 0,
 'average_characters_per_page': np.float64(1834.6)}

## 1.3 Document chunking

Complete pages can contain several unrelated topics. Retrieval works better when each searchable unit is focused.

`RecursiveCharacterTextSplitter` attempts to preserve larger boundaries such as paragraphs before splitting at finer boundaries. The selected chunk size and overlap are starting points rather than universal values.

In [11]:
def split_documents(
    documents: list[Document],
    *,
    chunk_size: int,
    chunk_overlap: int,
) -> list[Document]:
    """Split pages into retrieval-sized chunks while preserving metadata."""

    if chunk_overlap >= chunk_size:
        raise ValueError("chunk_overlap must be smaller than chunk_size")

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        # Store each chunk's starting character position in its metadata.
        add_start_index=True,
        # Prefer semantic boundaries before falling back to individual spaces.
        separators=["\n\n", "\n", ". ", " ", ""],
    )

    chunks = splitter.split_documents(documents)

    for index, chunk in enumerate(chunks):
        source = str(chunk.metadata["source"])
        page = int(chunk.metadata["page"])
        start_index = int(chunk.metadata.get("start_index", 0))

        # The stable ID connects a generated citation back to stored evidence.
        chunk.metadata["chunk_id"] = (
            f"{source}:page-{page}:start-{start_index}:chunk-{index}"
        )

    return chunks

In [12]:
chunks = split_documents(
    documents,
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

print(f"{len(documents)} pages became {len(chunks)} chunks")

9 pages became 29 chunks


In [13]:
# Inspect neighbouring chunks to see the effect of overlap.
for chunk in chunks[:2]:
    print("\nMetadata:", chunk.metadata)
    print("Page Content:")
    print(chunk.page_content[:600])


Metadata: {'source': 'support_escalation_guide.pdf', 'page': 1, 'start_index': 0, 'chunk_id': 'support_escalation_guide.pdf:page-1:start-0:chunk-0'}
Page Content:
AsterLane Delivery Services Ltd | ALS-SUP-002 | Version 1.4
Internal - Restricted | Page 1
AsterLane Delivery Services Ltd
Support Escalation Guide
Internal Knowledge Base - Fictional Training Material
Metadata field Value
Document ID ALS-SUP-002
Version 1.4
Effective date 1 June 2026
Review date 1 September 2026
Document owner Support Quality and Risk
Approved by Customer Operations Director
Audience Customer Support, Courier Support, Team Leaders
Region United Kingdom
Classification Internal - Restricted
Status Active
Tags escalation, safety, fraud, account security, privacy, 
complaints
Tra

Metadata: {'source': 'support_escalation_guide.pdf', 'page': 2, 'start_index': 0, 'chunk_id': 'support_escalation_guide.pdf:page-2:start-0:chunk-1'}
Page Content:
AsterLane Delivery Services Ltd | ALS-SUP-002 | Version 1.4
Internal - 

## 1.4 Text embeddings and semantic similarity

An embedding is a numeric representation of text. Texts with related meanings should have vectors that are close in the embedding space.

The same embedding model must be used for:

- document chunks during ingestion;
- user questions during retrieval.

In [14]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

example_sentences = [
    "A frontline agent may approve refunds up to seventy-five pounds.",
    "An eighty-pound refund needs additional approval.",
    "Couriers should attempt to contact an unavailable customer twice.",
]

example_vectors = embedding_model.encode(
    example_sentences,
    # Normalised vectors allow cosine similarity to be computed with a dot product.
    normalize_embeddings=True,
)

print("Embedding matrix shape:", example_vectors.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding matrix shape: (3, 384)


In [15]:
similarity_matrix = example_vectors @ example_vectors.T

for index, sentence in enumerate(example_sentences):
    print(f"\nSentence {index + 1}: {sentence}")
    print("Similarities:", np.round(similarity_matrix[index], 3))


Sentence 1: A frontline agent may approve refunds up to seventy-five pounds.
Similarities: [1.    0.612 0.104]

Sentence 2: An eighty-pound refund needs additional approval.
Similarities: [0.612 1.    0.187]

Sentence 3: Couriers should attempt to contact an unavailable customer twice.
Similarities: [0.104 0.187 1.   ]


## 1.5 Generating chunk embeddings

Embedding generation is performed in a batch for efficiency.

The result is one vector for every document chunk.

In [16]:
def embed_chunks(
    chunks: list[Document],
    model: SentenceTransformer,
) -> list[list[float]]:
    """Generate normalised embeddings for all document chunks."""

    texts = [chunk.page_content for chunk in chunks]

    vectors = model.encode(
        texts,
        normalize_embeddings=True,
        show_progress_bar=True,
    )

    return vectors.tolist()

In [17]:
chunk_embeddings = embed_chunks(chunks, embedding_model)

print("Number of chunk embeddings:", len(chunk_embeddings))
print("Embedding dimensions:", len(chunk_embeddings[0]))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Number of chunk embeddings: 29
Embedding dimensions: 384


In [18]:
# check an embedding
# chunk_embeddings[0]

## 1.6 Building the ChromaDB collection

Each ChromaDB record contains:

- a unique chunk ID;
- the chunk text;
- its embedding;
- source and page metadata.

For a repeatable classroom demonstration, the collection is rebuilt from scratch (reset: bool = True).
Incremental updating is outside this workshop's scope.

In [19]:
def build_vector_store(
    chunks: list[Document],
    embeddings: list[list[float]],
    *,
    chroma_dir: Path,
    collection_name: str,
    reset: bool = True,
):
    """Create a persistent Chroma collection from precomputed embeddings."""

    if not chunks:
        raise ValueError("At least one chunk is required.")

    if len(chunks) != len(embeddings):
        raise ValueError("Every chunk must have exactly one embedding.")

    dimensions = {len(embedding) for embedding in embeddings}
    if not dimensions or 0 in dimensions:
        raise ValueError("Embeddings must not be empty.")

    if len(dimensions) != 1:
        raise ValueError("All embeddings must have the same dimension.")

    ids = [str(chunk.metadata["chunk_id"]) for chunk in chunks]

    if len(ids) != len(set(ids)):
        raise ValueError("Every chunk must have a unique chunk_id.")



    # Create the folder for the persistent Chroma database.
    chroma_dir.mkdir(parents=True, exist_ok=True)
    
    # Connect to the existing database, or create it if needed.
    client = chromadb.PersistentClient(path=str(chroma_dir))

    if reset:
        # Delete only this collection so it can be rebuilt cleanly.
        existing_collections = [c.name for c in client.list_collections()]

        if collection_name in existing_collections:
            client.delete_collection(name=collection_name)

    

    # Open the named collection, or create it if it does not already exist.
    collection = client.get_or_create_collection(
        name=collection_name,
        # Use cosine distance for normalised Sentence Transformer embeddings.
        configuration={"hnsw": {"space": "cosine"}},
    )

    # Chroma expects one metadata dictionary for each stored chunk.
    metadatas = []

    for chunk in chunks:
        # Keep the source and character position so retrieved chunks can be
        # traced back to their original document.
        metadata = {
            "source": str(chunk.metadata["source"]),
            "start_index": int(chunk.metadata.get("start_index", 0)),
        }

        # Some document loaders provide page numbers, while others do not.
        # Only store the page when it is available.
        if "page" in chunk.metadata:
            metadata["page"] = int(chunk.metadata["page"])

        metadatas.append(metadata)

    # Insert new chunks, or replace existing chunks that have the same ID.
    collection.upsert(
        ids=ids,
        documents=[chunk.page_content for chunk in chunks],
        embeddings=embeddings,
        metadatas=metadatas,
    )

    # Return both objects so the caller can manage the database or query
    # this specific collection.
    return client, collection


In [20]:
chroma_client, collection = build_vector_store(
    chunks,
    chunk_embeddings,
    chroma_dir=CHROMA_DIR,
    collection_name=COLLECTION_NAME,
    reset=True,
)

print("Stored records:", collection.count())

Stored records: 29


## 1.7 Ingestion verification

The ingestion pipeline is complete when:

- the expected PDFs have been loaded;
- chunks contain source and page metadata;
- every chunk has one embedding;
- the ChromaDB collection contains the expected number of records.

In [21]:
assert len(documents)== 9
assert len(chunks) == len(chunk_embeddings)
assert collection.count() == len(chunks)
assert all("chunk_id" in chunk.metadata for chunk in chunks)
assert all("source" in chunk.metadata for chunk in chunks)
assert all("page" in chunk.metadata for chunk in chunks)

print("Ingestion verification passed.")

Ingestion verification passed.


# Part 2 — RAG Pipeline

The RAG pipeline runs for each user question.

RAG stands for **Retrieve, Augment, Generate**:

1. **Embed the question** — convert the user’s question into a vector.
2. **Retrieve** — find the most relevant chunks from the vector store.
3. **Augment** — add the retrieved chunks to the prompt as supporting context (i.e. augment the prompt).
4. **Generate** — use the language model to produce a structured, grounded answer.
5. **Validate** — check that the answer’s citations refer to the retrieved context.


## 2.1 Structured application schemas

Typed schemas make the system output predictable and easier to validate.

`RetrievedChunk` represents evidence returned by ChromaDB.  
`RAGAnswer` represents the structured response returned by the OpenAI agent.

In [23]:
class RetrievedChunk(BaseModel):
    chunk_id: str
    source: str
    page: int
    text: str
    distance: float


class Citation(BaseModel):
    chunk_id: str = Field(
        description="ID of a retrieved chunk supporting the answer."
    )
    source: str = Field(description="Source PDF filename.")
    page: int = Field(description="One-based PDF page number.")


class RAGAnswer(BaseModel):
    answer: str = Field(
        description="A concise answer based only on the retrieved context."
    )
    answerable: bool = Field(
        description="True only when the retrieved context contains enough evidence."
    )
    citations: list[Citation] = Field(
        default_factory=list,
        description="Evidence citations for the material claims in the answer.",
    )


class CitationCheck(BaseModel):
    valid: bool
    errors: list[str] = Field(default_factory=list)


class RAGResult(BaseModel):
    question: str
    answer: RAGAnswer
    retrieved_chunks: list[RetrievedChunk]
    citation_check: CitationCheck

## 2.2 [R]etrieve — semantic search

The user’s question is embedded with the same Sentence Transformer model used during ingestion.

ChromaDB compares the question embedding with the stored chunk embeddings and returns the `top_k` nearest matches.

This is the **Retrieve** stage of RAG. The retrieved chunks are inspected before they are added to the prompt or used to generate an answer.

In [24]:
def retrieve_context(
    question: str,
    *,
    collection,
    embedding_model: SentenceTransformer,
    top_k: int,
) -> list[RetrievedChunk]:
    """Retrieve the top-k chunks most relevant to a user question."""

    if not question.strip():
        raise ValueError("The question must not be empty.")

    # The query and documents must share the same embedding space.
    query_embedding = embedding_model.encode(
        question,
        normalize_embeddings=True,
    ).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )

    retrieved_chunks: list[RetrievedChunk] = []

    for chunk_id, text, metadata, distance in zip(
        results["ids"][0],
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
        strict=True,
    ):
        retrieved_chunks.append(
            RetrievedChunk(
                chunk_id=str(chunk_id),
                source=str(metadata["source"]),
                page=int(metadata["page"]),
                text=str(text),
                distance=float(distance),
            )
        )

    return retrieved_chunks

In [25]:
question = "Can a frontline support agent approve an £80 refund?"

retrieved_chunks = retrieve_context(
    question,
    collection=collection,
    embedding_model=embedding_model,
    top_k=TOP_K,
)


# how many chunks do you think we retrived?
len(retrieved_chunks)


4

In [26]:
# inspect each retrived chunck
for rank, chunk in enumerate(retrieved_chunks, start=1):
    print(
        f"\n[{rank}] Source:{chunk.source}; Page: {chunk.page}; "
        f"Distance:{chunk.distance:.4f}"
    )
    print("Content:")
    print(chunk.text)
    print("-"* 15)



[1] Source:customer_delivery_and_refund_policy.pdf; Page: 3; Distance:0.3407
Content:
items and must escalate the privacy incident.
7. Refund limits and exclusions
 Support agents may approve refunds up to £75 per order without team-leader approval, provided all policy 
conditions are met.
 Refunds above £75, multiple refunds on the same order, or goodwill credits above £15 require team-leader 
approval.
 AsterLane does not refund consequential losses, such as missed appointments, lost income or the cost of 
replacement travel.
 Promotional credits have no cash value and are normally returned as account credit rather than cash.
8. Customer communications
 Responses must explain the decision in plain language and identify the evidence used. Agents should avoid
---------------

[2] Source:support_escalation_guide.pdf; Page: 3; Distance:0.4332
Content:
£15.
 Amounts above these limits, multiple refunds on one order, or exceptions to published policy require team-
leader approval.


### Retrieval inspection

Before generation, verify that the retrieved context is suitable.

Inspect:

- source documents;
- ranking;
- distance values;
- whether the text contains the required evidence;
- whether irrelevant material has been included.

Distance is a similarity signal, not a calibrated probability of correctness.

In [27]:
retrieved_sources = [chunk.source for chunk in retrieved_chunks]
expected_source = "customer_delivery_and_refund_policy.pdf"

print("Retrieved sources:", retrieved_sources)
print("Expected source present:", expected_source in retrieved_sources)

Retrieved sources: ['customer_delivery_and_refund_policy.pdf', 'support_escalation_guide.pdf', 'support_escalation_guide.pdf', 'customer_delivery_and_refund_policy.pdf']
Expected source present: True


## 2.3 [A]ugment — grounded prompt construction

The prompt contains:

- clear behavioural instructions;
- labelled retrieved chunks;
- the user's question;
- requirements for abstention and citations.

Chunk identifiers are exposed to the model so citations can be validated afterwards.

In [28]:
def build_context(chunks: list[RetrievedChunk]) -> str:
    """Format retrieved chunks as labelled prompt context."""

    context_sections: list[str] = []

    for chunk in chunks:
        header = (
            f"[chunk_id={chunk.chunk_id} | "
            f"source={chunk.source} | page={chunk.page}]"
        )

        context_sections.append(f"{header}\n{chunk.text}")

    return "\n\n---\n\n".join(context_sections)



RAG_PROMPT_TEMPLATE = """
Use only the retrieved context below to answer the user question.

Rules:
- Do not use outside knowledge.
- If the context is insufficient, set answerable to false.
- If answerable is true, cite the retrieved chunks supporting material claims.
- Use chunk_id, source and page exactly as provided.

RETRIEVED CONTEXT
{context}

USER QUESTION
{question}
"""


def build_rag_prompt(
    question: str,
    chunks: list[RetrievedChunk],
) -> str:
    """Fill the RAG prompt template with retrieved context and the question."""

    context = build_context(chunks)

    return RAG_PROMPT_TEMPLATE.format(
        context=context,
        question=question,
    )

In [30]:
rag_prompt = build_rag_prompt(question, retrieved_chunks)
print(rag_prompt)


Use only the retrieved context below to answer the user question.

Rules:
- Do not use outside knowledge.
- If the context is insufficient, set answerable to false.
- If answerable is true, cite the retrieved chunks supporting material claims.
- Use chunk_id, source and page exactly as provided.

RETRIEVED CONTEXT
[chunk_id=customer_delivery_and_refund_policy.pdf:page-3:start-693:chunk-17 | source=customer_delivery_and_refund_policy.pdf | page=3]
items and must escalate the privacy incident.
7. Refund limits and exclusions
 Support agents may approve refunds up to £75 per order without team-leader approval, provided all policy 
conditions are met.
 Refunds above £75, multiple refunds on the same order, or goodwill credits above £15 require team-leader 
approval.
 AsterLane does not refund consequential losses, such as missed appointments, lost income or the cost of 
replacement travel.
 Promotional credits have no cash value and are normally returned as account credit rather than 

## 2.4 [G]enerate — structured answer generation

The grounded prompt is passed to an OpenAI agent configured with `RAGAnswer` as its output type.

This is the **Generate** stage of RAG. The model produces a structured response containing:

- the grounded answer;
- whether the question is answerable from the retrieved context;
- citations supporting the answer.

Because `output_type=RAGAnswer` is configured, the final output is parsed and validated as a `RAGAnswer` object rather than returned as unstructured text.

In [31]:
def create_answer_agent(model_name: str) -> Agent:
    """Create the grounded answer agent with a structured output schema."""

    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError(
            "OPENAI_API_KEY is missing. Add it to the project .env file."
        )

    return Agent(
        name="AsterLane grounded policy assistant",
        model=model_name,
        instructions=(
            "Answer questions about the supplied AsterLane documents. "
            "Use only the retrieved context. "
            "Return a concise answer with verifiable citations."
        ),
        output_type=RAGAnswer,
    )


async def generate_answer(
    question: str,
    chunks: list[RetrievedChunk],
    *,
    model_name: str,
) -> RAGAnswer:
    """Generate a structured answer grounded in the retrieved chunks."""

    agent = create_answer_agent(model_name)
    prompt = build_rag_prompt(question, chunks)

    run = await Runner.run(agent, prompt)

    # When output_type is configured, final_output is parsed into RAGAnswer.
    output = run.final_output

    if not isinstance(output, RAGAnswer):
        raise TypeError("The agent did not return a RAGAnswer object.")

    return output

In [33]:
structured_answer = await generate_answer(
    question,
    retrieved_chunks,
    model_name=OPENAI_MODEL,
)

#structured_answer
pprint(structured_answer.model_dump())

{'answer': 'No, a frontline support agent cannot approve an £80 refund without '
           'team-leader approval, as the limit for refunds they can approve is '
           'up to £75 per order.',
 'answerable': True,
 'citations': [{'chunk_id': 'customer_delivery_and_refund_policy.pdf:page-3:start-693:chunk-17',
                'page': 3,
                'source': 'customer_delivery_and_refund_policy.pdf'},
               {'chunk_id': 'support_escalation_guide.pdf:page-3:start-682:chunk-7',
                'page': 3,
                'source': 'support_escalation_guide.pdf'}]}


In [36]:
# PUTTING IT ALL TOGETHER

courier_question = "Can couriers accept a cash payment for delivery to a different address.?"


retrieved_chunks = retrieve_context(
    courier_question,
    collection=collection,
    embedding_model=embedding_model,
    top_k=TOP_K,
)


structured_answer = await generate_answer(
    courier_question,
    retrieved_chunks,
    model_name=OPENAI_MODEL,
)

#structured_answer
pprint(structured_answer.model_dump())

{'answer': 'No, couriers must not accept a cash payment or private arrangement '
           'to deliver to a different address.',
 'answerable': True,
 'citations': [{'chunk_id': 'courier_operations_handbook.pdf:page-3:start-577:chunk-26',
                'page': 3,
                'source': 'courier_operations_handbook.pdf'}]}


In [41]:
#retrieved_chunks

## 2.6 Validate — citation checking

Structured output guarantees the shape of a citation, but it does not prove that the citation is correct.

The validation step checks that:

- the cited chunk was retrieved;
- the source and page match the retrieved chunk;
- the quotation appears in that chunk;
- answerable responses include supporting citations;
- unanswerable responses do not claim supporting citations.

In [42]:
def validate_citations(
    answer: RAGAnswer,
    retrieved_chunks: list[RetrievedChunk],
) -> CitationCheck:
    """Validate model-generated citations against retrieved evidence."""

    # Build a fast lookup from chunk ID to retrieved chunk.
    chunk_lookup = {
        chunk.chunk_id: chunk
        for chunk in retrieved_chunks
    }

    errors: list[str] = []

    if answer.answerable and not answer.citations:
        errors.append(
            "The answer is marked answerable but contains no citations."
        )

    if not answer.answerable and answer.citations:
        errors.append(
            "An unanswerable response should not contain evidence citations."
        )

    for citation in answer.citations:
        chunk = chunk_lookup.get(citation.chunk_id)

        if chunk is None:
            errors.append(
                f"The cited chunk was not retrieved: {citation.chunk_id}"
            )
            continue

        if citation.source != chunk.source:
            errors.append(
                f"Source mismatch for chunk {citation.chunk_id}"
            )

        if citation.page != chunk.page:
            errors.append(
                f"Page mismatch for chunk {citation.chunk_id}"
            )

    # The citation check is valid only when no validation errors were found.
    return CitationCheck(
        valid=not errors,
        errors=errors,
    )

In [43]:
citation_check = validate_citations(
    structured_answer,
    retrieved_chunks,
)

# citation_check
print(citation_check.model_dump_json(indent=2))

{
  "valid": true,
  "errors": []
}


## 2.7 End-to-end RAG pipeline

The complete function composes the request-time stages of the pipeline:

1. **Retrieve** the most relevant chunks.
2. **Augment and generate** a structured answer from the retrieved evidence.
3. **Validate** the generated citations.

Returning the retrieved chunks alongside the answer supports debugging, inspection and evaluation.

In [44]:
async def answer_question(
    question: str,
    *,
    collection,
    embedding_model: SentenceTransformer,
    model_name: str,
    top_k: int = 4,
) -> RAGResult:
    """Run the complete request-time RAG pipeline."""

    retrieved = retrieve_context(
        question,
        collection=collection,
        embedding_model=embedding_model,
        top_k=top_k,
    )

    answer = await generate_answer(
        question,
        retrieved,
        model_name=model_name,
    )

    citation_check = validate_citations(
        answer,
        retrieved,
    )

    return RAGResult(
        question=question,
        answer=answer,
        retrieved_chunks=retrieved,
        citation_check=citation_check,
    )

In [46]:
complete_result = await answer_question(
    question,
    collection=collection,
    embedding_model=embedding_model,
    model_name=OPENAI_MODEL,
    top_k=TOP_K,
)

print(complete_result.model_dump_json(indent=2))

{
  "question": "Can a frontline support agent approve an £80 refund?",
  "answer": {
    "answer": "No, a frontline support agent cannot approve an £80 refund. Refunds above £75 require team-leader approval.",
    "answerable": true,
    "citations": [
      {
        "chunk_id": "customer_delivery_and_refund_policy.pdf:page-3:start-693:chunk-17",
        "source": "customer_delivery_and_refund_policy.pdf",
        "page": 3
      },
      {
        "chunk_id": "support_escalation_guide.pdf:page-3:start-682:chunk-7",
        "source": "support_escalation_guide.pdf",
        "page": 3
      }
    ]
  },
  "retrieved_chunks": [
    {
      "chunk_id": "customer_delivery_and_refund_policy.pdf:page-3:start-693:chunk-17",
      "source": "customer_delivery_and_refund_policy.pdf",
      "page": 3,
      "text": "items and must escalate the privacy incident.\n7. Refund limits and exclusions\n Support agents may approve refunds up to £75 per order without team-leader approval, provided all p

In [43]:
medical_question = "What if a customer calls you with medical emergency?"

complete_result = await answer_question(
    medical_question,
    collection=collection,
    embedding_model=embedding_model,
    model_name=OPENAI_MODEL,
    top_k=TOP_K,
)

print(complete_result.model_dump_json(indent=2))

{
  "question": "What if a customer calls you with medical emergency?",
  "answer": {
    "answer": "If a customer calls with a medical emergency, you should advise them to contact the appropriate emergency service immediately. After ensuring immediate safety action, you must create a Priority 1 incident and notify the Duty Operations Manager.",
    "answerable": true,
    "citations": [
      {
        "chunk_id": "support_escalation_guide.pdf:page-2:start-0:chunk-1",
        "source": "support_escalation_guide.pdf",
        "page": 2
      }
    ]
  },
  "retrieved_chunks": [
    {
      "chunk_id": "support_escalation_guide.pdf:page-2:start-0:chunk-1",
      "source": "support_escalation_guide.pdf",
      "page": 2,
      "text": "AsterLane Delivery Services Ltd | ALS-SUP-002 | Version 1.4\nInternal - Restricted | Page 2\n1. Purpose\n This guide defines when a frontline support agent may resolve a case and when the case must be passed to a \nspecialist queue, team leader or emergen

# Part 3 — Testing and Evaluation

RAG systems should be evaluated by pipeline stage rather than judged only by the final answer.

This notebook uses five evaluation layers:

1. **Ingestion checks** — was the knowledge base built correctly?
2. **Retrieval evaluation** — did the system retrieve the required evidence?
3. **Generation and citation evaluation** — was the answer grounded and correctly attributed?
4. **End-to-end evaluation** — did the complete system produce the expected result?
5. **LLM-as-a-judge** — does the generated answer semantically agree with the reference answer and retrieved evidence?

## 3.1 Golden evaluation dataset

A golden dataset contains representative questions together with human-defined expected results.

Each case records:

- the evaluation question;
- a reference answer;
- the expected source documents;
- whether the question should be answerable;
- the type of evaluation case.

Useful evaluation slices include:

- direct factual questions;
- paraphrased questions;
- policy-boundary questions;
- multi-source questions;
- unanswerable questions.


> NOTE: The reference answer provides the expected meaning of a correct response. The generated answer does not need to match it word-for-word.Later, an LLM judge will compare the generated and reference answers semantically.

In [47]:
EVAL_CASES = [
    {
        "id": "refund_limit",
        "question": "Can a frontline support agent approve an £80 refund?",
        "reference_answer": (
            "No. Refunds above £75 require additional approval."
        ),
        "expected_sources": [
            "customer_delivery_and_refund_policy.pdf",
        ],
        "answerable": True,
        "slice": "policy_threshold",
    },
    {
        "id": "contact_attempts",
        "question": (
            "What should a courier do when the customer cannot be reached?"
        ),
        "reference_answer": (
            "The courier should make the required contact attempts and wait "
            "for the specified period before marking the customer unavailable."
        ),
        "expected_sources": [
            "courier_operations_handbook.pdf",
        ],
        "answerable": True,
        "slice": "paraphrase",
    },
    {
        "id": "courier_pay",
        "question": "How much is a courier paid for each delivery?",
        "reference_answer": None,
        "expected_sources": [],
        "answerable": False,
        "slice": "unanswerable",
    },
        {
        "id": "irrelevent_question",
        "question": "What is the stock value of your company?",
        "reference_answer": None,
        "expected_sources": [],
        "answerable": False,
        "slice": "unanswerable",
    },
]

EVAL_CASES

[{'id': 'refund_limit',
  'question': 'Can a frontline support agent approve an £80 refund?',
  'reference_answer': 'No. Refunds above £75 require additional approval.',
  'expected_sources': ['customer_delivery_and_refund_policy.pdf'],
  'answerable': True,
  'slice': 'policy_threshold'},
 {'id': 'contact_attempts',
  'question': 'What should a courier do when the customer cannot be reached?',
  'reference_answer': 'The courier should make the required contact attempts and wait for the specified period before marking the customer unavailable.',
  'expected_sources': ['courier_operations_handbook.pdf'],
  'answerable': True,
  'slice': 'paraphrase'},
 {'id': 'courier_pay',
  'question': 'How much is a courier paid for each delivery?',
  'reference_answer': None,
  'expected_sources': [],
  'answerable': False,
  'slice': 'unanswerable'},
 {'id': 'irrelevent_question',
  'question': 'What is the stock value of your company?',
  'reference_answer': None,
  'expected_sources': [],
  'answ

## 3.2 Ingestion checks

These deterministic checks verify that each stage of ingestion completed successfully before retrieval is attempted.

In [48]:
def evaluate_ingestion(
    documents: list[Document],
    chunks: list[Document],
    embeddings: list[list[float]],
    collection,
) -> dict[str, bool]:
    """Run deterministic checks over the ingestion pipeline."""

    documents_loaded = len(documents) > 0
    chunks_created = len(chunks) > 0
    one_embedding_per_chunk = len(chunks) == len(embeddings)
    collection_count_matches = collection.count() == len(chunks)

    chunk_ids_present = all(
        "chunk_id" in chunk.metadata
        for chunk in chunks
    )

    source_metadata_present = all(
        "source" in chunk.metadata
        for chunk in chunks
    )

    page_metadata_present = all(
        "page" in chunk.metadata
        for chunk in chunks
    )

    return {
        "documents_loaded": documents_loaded,
        "chunks_created": chunks_created,
        "one_embedding_per_chunk": one_embedding_per_chunk,
        "collection_count_matches": collection_count_matches,
        "chunk_ids_present": chunk_ids_present,
        "source_metadata_present": source_metadata_present,
        "page_metadata_present": page_metadata_present,
    }

In [49]:
ingestion_results = evaluate_ingestion(
    documents,
    chunks,
    chunk_embeddings,
    collection,
)

ingestion_results

{'documents_loaded': True,
 'chunks_created': True,
 'one_embedding_per_chunk': True,
 'collection_count_matches': True,
 'chunk_ids_present': True,
 'source_metadata_present': True,
 'page_metadata_present': True}

In [50]:
all_checks_passed = all(ingestion_results.values())

print("All ingestion checks passed:", all_checks_passed)

All ingestion checks passed: True


## 3.3 Retrieval metrics

Retrieval metrics measure whether the system found the correct evidence and how highly that evidence was ranked.

Suppose the expected source appears in the retrieved results at rank 3:

1. unrelated source
2. unrelated source
3. expected source
4. unrelated source

The notebook uses three simple retrieval metrics.

### 1. Hit@k

**Hit@k** asks:

> Did at least one expected source appear anywhere in the top `k` retrieved results?

- returns `1` when an expected source is found;
- returns `0` when no expected source is found.

In the example above:

```text
Hit@4 = 1
```

because the expected source appears within the top four results.

### 2. First relevant rank

The **first relevant rank** is the position of the first expected source in the retrieved results.

In the example above:

```text
first relevant rank = 3
```

A lower rank is better because the correct evidence appears earlier in the results.

### 3. Reciprocal rank

**Reciprocal rank** converts the first relevant rank into a score:

```text
reciprocal rank = 1 / rank
```

Examples:

| First relevant rank | Reciprocal rank |
|---:|---:|
| 1 | 1.00 |
| 2 | 0.50 |
| 3 | 0.33 |
| 4 | 0.25 |
| not found | 0.00 |

A higher reciprocal rank is better.

These metrics evaluate retrieval only. They do not assess whether the generated answer is correct.


In [51]:
def hit_at_k(
    retrieved: list[RetrievedChunk],
    expected_sources: list[str],
    *,
    k: int,
) -> int:
    """Return 1 if an expected source appears in the top-k results."""

    expected = set(expected_sources)
    top_k_chunks = retrieved[:k]

    for chunk in top_k_chunks:
        if chunk.source in expected:
            return 1

    return 0

def first_relevant_rank(
    retrieved: list[RetrievedChunk],
    expected_sources: list[str],
) -> int | None:
    """Return the one-based rank of the first expected source."""

    expected = set(expected_sources)

    for rank, chunk in enumerate(retrieved, start=1):
        if chunk.source in expected:
            return rank

    return None


def reciprocal_rank(
    retrieved: list[RetrievedChunk],
    expected_sources: list[str],
) -> float:
    """Return the reciprocal rank of the first expected source."""

    rank = first_relevant_rank(
        retrieved,
        expected_sources,
    )

    return 0.0 if rank is None else 1.0 / rank

In [52]:
retrieval_case = EVAL_CASES[0]

case_chunks = retrieve_context(
    retrieval_case["question"],
    collection=collection,
    embedding_model=embedding_model,
    top_k=TOP_K,
)

hit = hit_at_k(
    case_chunks,
    retrieval_case["expected_sources"],
    k=TOP_K,
)

first_rank = first_relevant_rank(
    case_chunks,
    retrieval_case["expected_sources"],
)

rr = reciprocal_rank(
    case_chunks,
    retrieval_case["expected_sources"],
)

print("Hit@k:", hit)
print("First relevant rank:", first_rank)
print("Reciprocal rank:", round(rr, 3))

Hit@k: 1
First relevant rank: 1
Reciprocal rank: 1.0


The retrieval succeeded: an expected source appeared in the top results and was ranked first. This gives a reciprocal rank of `1.0`, the highest possible score.

## 3.4 Answerability and abstention

A RAG system should answer only when the retrieved evidence is sufficient.

An answerability check compares:

- whether the question is expected to be answerable from the knowledge base;
- whether the system chose to answer.

This produces four possible outcomes:

| Evidence exists | System behaviour | Interpretation |
|---|---|---|
| Yes | Answers | Correct answering |
| Yes | Refuses | False refusal |
| No | Refuses | Correct abstention |
| No | Answers | Unsupported answer |

The final case is especially important because the model may be guessing or relying on prior knowledge rather than the retrieved evidence.

In [53]:
def classify_answerability(
    *,
    expected_answerable: bool,
    predicted_answerable: bool,
) -> str:
    """Classify whether the system answered or abstained correctly."""

    if expected_answerable and predicted_answerable:
        return "correct_answering"

    if expected_answerable and not predicted_answerable:
        return "false_refusal"

    if not expected_answerable and not predicted_answerable:
        return "correct_abstention"

    return "unsupported_answer"

## 3.5 Citation and groundedness checks

Citation validation checks whether the generated answer is supported by the retrieved evidence.

For each answer, inspect:

- whether the citations are valid;
- whether the cited source and page are correct;
- whether the quoted text appears in the cited chunk;
- whether important claims are supported;
- whether the answer introduces unsupported claims.

The first three checks can be automated directly.

> Checking whether every important claim is fully supported may require human review or a separate model-based evaluator.

> The  LLM-as-a-judge section later in the notebook demonstrates one way to perform this semantic evaluation.

In [54]:
# Example of deterministic citation evaluation:
#
result = await answer_question(
    EVAL_CASES[0]["question"],
    collection=collection,
    embedding_model=embedding_model,
    model_name=OPENAI_MODEL,
    top_k=TOP_K,
)

print("Citations valid:", result.citation_check.valid)
print("Citation errors:", result.citation_check.errors)

Citations valid: True
Citation errors: []


## 3.6 End-to-end evaluation

End-to-end evaluation checks the complete RAG workflow for one golden test case.

It records both retrieval and generation behaviour, including:

- the evaluation case ID;
- the evaluation slice;
- whether an expected source appeared in the retrieved results;
- the rank of the first expected source;
- whether the system answered or abstained correctly;
- whether the citations passed validation;
- the generated structured answer.

This makes it easier to diagnose whether a failure came from retrieval, generation, answerability or citation handling.

These checks do not yet determine whether the generated answer is semantically equivalent to the reference answer. That is explored later using an LLM-as-a-judge.

In [55]:
async def evaluate_case(
    case: dict[str, Any],
    *,
    collection,
    embedding_model: SentenceTransformer,
    model_name: str,
    top_k: int,
) -> dict[str, Any]:
    """Evaluate one golden-set case across retrieval and generation."""

    result = await answer_question(
        case["question"],
        collection=collection,
        embedding_model=embedding_model,
        model_name=model_name,
        top_k=top_k,
    )

    expected_sources = case["expected_sources"]

    if expected_sources:
        hit = hit_at_k(
            result.retrieved_chunks,
            expected_sources,
            k=top_k,
        )

        first_rank = first_relevant_rank(
            result.retrieved_chunks,
            expected_sources,
        )
    else:
        hit = None
        first_rank = None

    answerability_outcome = classify_answerability(
        expected_answerable=case["answerable"],
        predicted_answerable=result.answer.answerable,
    )

    return {
        "id": case["id"],
        "slice": case["slice"],
        "hit_at_k": hit,
        "first_relevant_rank": first_rank,
        "answerability_outcome": answerability_outcome,
        "citations_valid": result.citation_check.valid,
        "answer": result.answer.answer,
    }

In [56]:

evaluation_result = await evaluate_case(
    EVAL_CASES[0],
    collection=collection,
    embedding_model=embedding_model,
    model_name=OPENAI_MODEL,
    top_k=TOP_K,
)

evaluation_result

{'id': 'refund_limit',
 'slice': 'policy_threshold',
 'hit_at_k': 1,
 'first_relevant_rank': 1,
 'answerability_outcome': 'correct_answering',
 'citations_valid': True,
 'answer': 'No, a frontline support agent cannot approve an £80 refund; they can only approve refunds up to £75 without team-leader approval. Refunds above £75 require team-leader approval.'}

In [57]:

evaluation_result = await evaluate_case(
    EVAL_CASES[-1],
    collection=collection,
    embedding_model=embedding_model,
    model_name=OPENAI_MODEL,
    top_k=TOP_K,
)

evaluation_result

{'id': 'irrelevent_question',
 'slice': 'unanswerable',
 'hit_at_k': None,
 'first_relevant_rank': None,
 'answerability_outcome': 'correct_abstention',
 'citations_valid': True,
 'answer': 'The retrieved context does not provide any information regarding the stock value of AsterLane Delivery Services Ltd.'}

This evaluation case passed across retrieval, answerability and citation validation. The expected source was ranked first, and the generated answer correctly reflected the £75 approval limit.

## 3.7 Controlled degradation test [OPTIONAL]

A controlled degradation test changes one setting while keeping the rest of the pipeline fixed.

For example, reduce `top_k` from four to one and inspect whether the relevant evidence disappears.
This helps identify how sensitive the system is to retrieval settings.

The diagnostic order is:

```text
document extraction
-> chunking
-> retrieval
-> prompt
-> generation
-> citations
```

In [58]:
baseline_chunks = retrieve_context(
    question,
    collection=collection,
    embedding_model=embedding_model,
    top_k=4,
)

degraded_chunks = retrieve_context(
    question,
    collection=collection,
    embedding_model=embedding_model,
    top_k=1,
)

print("Baseline retrieved:", len(baseline_chunks))
print("Degraded retrieved:", len(degraded_chunks))

Baseline retrieved: 4
Degraded retrieved: 1


## 3.8 Optional extension — LLM-as-a-judge

Deterministic checks are useful when correctness can be tested with exact rules.

For example, this notebook can check whether:

- an expected source was retrieved;
- a citation points to a retrieved chunk;
- the citation source and page match;
- the system answered or abstained.

However, deterministic checks are less effective for questions such as:

- Does the generated answer mean the same thing as the reference answer?
- Is the answer fully supported by the retrieved evidence?
- Did the answer omit an important condition?
- Is the answer concise and relevant?

Two answers can be semantically equivalent even when they use different words.

For example:

> Reference answer: Refunds above £75 require team-leader approval.

> Generated answer: A frontline agent cannot approve an £80 refund without approval from a team leader.

An exact string comparison would treat these as different, even though their meaning is the same.

### What is an LLM-as-a-judge?

An **LLM-as-a-judge** uses a language model to evaluate the output of another language-model call.

The judge receives information such as:

- the user question;
- the reference answer;
- the generated answer;
- the retrieved context;
- whether the question was expected to be answerable.

It then produces a structured judgement.

In this notebook, the judge will assess:

1. **Correctness** — does the generated answer agree with the reference answer?
2. **Groundedness** — is the generated answer supported by the retrieved context?
3. **Relevance** — does the answer directly address the question?
4. **Overall quality** — how good is the answer on a scale from 1 to 5?

### Why not use an LLM judge for everything?

LLM judges are useful, but they are not objective ground truth.

Their judgements may be affected by:

- the wording of the evaluation prompt;
- the model used as the judge;
- answer length and writing style;
- incomplete reference answers;
- randomness between runs.

They also add extra cost and latency.

A sensible evaluation strategy is therefore:

1. use deterministic checks wherever possible;
2. use an LLM judge for semantic qualities;
3. compare judge results with human-labelled examples;
4. retain human review for important or ambiguous cases.

In [59]:
class JudgeResult(BaseModel):
    correct: bool = Field(
        description=(
            "True when the generated answer agrees with the reference answer "
            "and does not contradict it."
        )
    )
    grounded: bool = Field(
        description=(
            "True when every material claim in the generated answer is "
            "supported by the retrieved context."
        )
    )
    relevant: bool = Field(
        description=(
            "True when the generated answer directly addresses the user question."
        )
    )
    score: int = Field(
        ge=1,
        le=5,
        description="Overall answer-quality score from 1 to 5.",
    )
    explanation: str = Field(
        description="A brief explanation of the judgement."
    )

The judge returns structured output rather than free-form text.

Pydantic ensures that:

- `correct`, `grounded` and `relevant` are Boolean values;
- `score` is between 1 and 5;
- an explanation is returned.

Pydantic validates the shape of the result. The language model still decides the judgement itself.

In [60]:
JUDGE_PROMPT_TEMPLATE = """Evaluate the generated RAG answer.

Use the reference answer to assess correctness.
Use the retrieved context to assess groundedness.
Do not reward writing style unless it improves clarity.

Evaluation criteria:

1. Correctness
- The generated answer should agree with the reference answer.
- Different wording is acceptable.
- Important conditions or limits must not be omitted.
- A contradiction should be marked incorrect.

2. Groundedness
- Every material claim must be supported by the retrieved context.
- The answer must not rely on outside knowledge.

3. Relevance
- The answer should directly address the user question.
- Irrelevant detail should not improve the score.

4. Overall score
- 5: fully correct, grounded and relevant
- 4: correct overall, with a minor omission or clarity issue
- 3: partly correct, but with a meaningful omission or unsupported detail
- 2: substantially incorrect or poorly grounded
- 1: incorrect, unsupported or unrelated

USER QUESTION
{question}

REFERENCE ANSWER
{reference_answer}

GENERATED ANSWER
{generated_answer}

RETRIEVED CONTEXT
{context}
"""

In [61]:
def create_judge_agent(model_name: str) -> Agent:
    """Create an agent that evaluates a generated RAG answer."""

    return Agent(
        name="RAG answer judge",
        instructions=(
            "Evaluate the answer using only the supplied question, "
            "reference answer and retrieved context. "
            "Return a strict and concise judgement."
        ),
        model=model_name,
        output_type=JudgeResult,
    )

In [62]:
async def judge_rag_answer(
    *,
    question: str,
    reference_answer: str,
    generated_answer: str,
    retrieved_chunks: list[RetrievedChunk],
    model_name: str,
) -> JudgeResult:
    """Evaluate answer correctness, groundedness and relevance."""

    context = build_context(retrieved_chunks)

    prompt = JUDGE_PROMPT_TEMPLATE.format(
        question=question,
        reference_answer=reference_answer,
        generated_answer=generated_answer,
        context=context,
    )

    judge_agent = create_judge_agent(model_name)
    run = await Runner.run(judge_agent, prompt)

    return run.final_output

In [63]:
case = EVAL_CASES[0]

rag_result = await answer_question(
    case["question"],
    collection=collection,
    embedding_model=embedding_model,
    model_name=OPENAI_MODEL,
    top_k=TOP_K,
)

judge_result = await judge_rag_answer(
    question=case["question"],
    reference_answer=case["reference_answer"],
    generated_answer=rag_result.answer.answer,
    retrieved_chunks=rag_result.retrieved_chunks,
    model_name=OPENAI_MODEL,
)

print(judge_result.model_dump_json(indent=2))

{
  "correct": true,
  "grounded": true,
  "relevant": true,
  "score": 5,
  "explanation": "The generated answer is fully correct, grounded in the provided context, and directly addresses the user question. It accurately conveys the requirement for additional approval for refunds above £75, which aligns with the reference answer."
}


The judge rated the answer as fully correct, grounded and relevant.

This demonstrates why an LLM judge can be useful: the generated answer did not need to match the reference answer word-for-word. The judge assessed whether the meaning was equivalent and whether the answer was supported by the retrieved context.

> **Important:** An LLM judge provides a model-based assessment, not objective ground truth. Its score may vary with the judge model, prompt wording or evaluation run. Use judge results alongside deterministic checks and, where appropriate, human review.

## 3.9 From notebook implementation to source modules

The notebook has now built every function directly.

The same functions are organised into clean modules for reuse:

```text
src/rag_workshop/
├── ingestion.py   # loading, splitting, embedding and storage
├── rag.py         # retrieval, prompt construction and generation
├── validation.py  # citation validation
├── evaluation.py  # retrieval metrics
├── schemas.py     # Pydantic application models
└── config.py      # shared configuration
```

Student-facing commands:

```bash
poetry run python scripts/ingest.py
poetry run python scripts/ask.py "Your question" --show-context
```

The source files are a refactoring of the notebook implementation, not a separate design.

# Summary

The workshop has implemented three layers:

```text
PART 1 — INGESTION
load -> inspect -> split -> embed -> store

PART 2 — RAG
embed query -> retrieve -> build prompt
            -> generate structured answer -> validate citations

PART 3 — TESTING & EVALUATION
ingestion checks -> retrieval metrics -> answerability
                 -> citation checks -> end-to-end evaluation
```

The central engineering principle is:

> Evaluate each stage independently before evaluating the complete system.